### Phase 2.1a: Parsing the JSON Structure (Severity Dataset)
Here we load the `.jsonl` files containing the new severity data into a pandas DataFrame. We use regular expressions to parse the filename (e.g., `Bearing_fault_shaftend_10mins_2000rpm_sev1.jsonl`) to automatically extract the `machine_state`, `motor_speed`, and `severity` level, adding them as new columns to our dataset.

In [ ]:
import os
import json
import pandas as pd
import glob
import re
import zipfile

data_source = "/content/final data for severity based modelling.zip"
extract_dir = "/content/extracted_severity_data"

# Extract zip if necessary
if data_source.endswith('.zip'):
    if not os.path.exists(extract_dir):
        print(f"Extracting {data_source}...")
        with zipfile.ZipFile(data_source, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
    # Use recursive glob in case files are inside a nested folder in the zip
    jsonl_files = glob.glob(os.path.join(extract_dir, "**/*.jsonl"), recursive=True)
else:
    jsonl_files = glob.glob(os.path.join(data_source, "*.jsonl"))

all_severity_data = []

for file_path in jsonl_files:
    filename = os.path.basename(file_path)

    # Template: FaultName_Position_Duration_RPM_Severity_Extra.jsonl
    # Example: MechanicalLooseness_Shaftend_10mins_2200rpm_high_360deg.jsonl

    # Use regex to extract the parts safely.
    match = re.search(r'(.+?)_.*?_\d+mins_(\d+rpm)_([a-z0-9]+)_.*\.jsonl$', filename, re.IGNORECASE)

    if match:
        # Group 1: Fault_name, Group 2: motorrpm, Group 3: severitylevel
        state = match.group(1).replace('_', ' ')
        motor_speed = match.group(2).lower()
        severity = match.group(3).lower()
    else:
        # Fallback if a file doesn't perfectly match the new template
        lower_name = filename.lower()
        if '_shaftend' in lower_name:
            state_part = filename[:lower_name.find('_shaftend')]
        elif '_10mins' in lower_name:
            state_part = filename[:lower_name.find('_10mins')]
        else:
            state_part = filename.split('_')[0]

        state = state_part.replace('_', ' ')

        sev_match = re.search(r'_(baseline|low|medium|high|sev\d+)_', lower_name)
        severity = sev_match.group(1) if sev_match else 'unknown'

        rpm_match = re.search(r'(\d+rpm)', lower_name)
        motor_speed = rpm_match.group(1) if rpm_match else 'unknown'

    # Force 'baseline' severity for normal machine state
    if state.lower().strip() == 'normal':
        severity = 'baseline'

    with open(file_path, 'r') as f:
        for line in f:
            try:
                row_data = json.loads(line.strip())
                row_data['machine_state'] = state
                row_data['severity'] = severity
                row_data['motor_speed'] = motor_speed
                all_severity_data.append(row_data)
            except json.JSONDecodeError:
                continue # Skip invalid lines

# Convert to a pandas DataFrame
df_severity = pd.DataFrame(all_severity_data)

print(f"Total rows loaded: {len(df_severity)}")
if not df_severity.empty:
    print(f"Unique Severity Levels Found: {df_severity['severity'].unique()}")
    print(f"Unique Motor Speeds Found: {df_severity['motor_speed'].unique()}")
    display(df_severity.head())
else:
    print("No data found. Please double check the data_source path or the contents of the zip file!")

Extracting /content/final data for severity based modelling.zip...
Total rows loaded: 73363
Unique Severity Levels Found: ['medium' 'high' 'baseline' 'low']
Unique Motor Speeds Found: ['3200rpm' '2700rpm' '2200rpm']


,time,axis,data,machine_state,severity,motor_speed
0,1.777510e+09,Z,"[-455.4404296875, -572.4404296875, -483.440429...",MechanicalLooseness,medium,3200rpm
1,1.777510e+09,X,"[-173.169921875, -229.169921875, -241.16992187...",MechanicalLooseness,medium,3200rpm
2,1.777510e+09,Y,"[114.1552734375, 153.1552734375, 165.155273437...",MechanicalLooseness,medium,3200rpm
3,1.777510e+09,Z,"[765.4775390625, 549.4775390625, 316.477539062...",MechanicalLooseness,medium,3200rpm
4,1.777510e+09,X,"[-466.0517578125, -548.0517578125, -630.051757...",MechanicalLooseness,medium,3200rpm


In [ ]:
import pandas as pd

# Create a stratified small sample (e.g., 10 rows per combination of state, severity, and speed)
# This ensures your sample has a good mix of all fault types and baselines
df_sample = df_severity.groupby(
    ['machine_state', 'severity', 'motor_speed'],
    group_keys=False
).apply(lambda x: x.sample(min(len(x), 20), random_state=42), include_groups=False).reset_index(drop=True)

# Save the sample to a JSON Lines file to preserve the list arrays in the 'data' column
sample_path = '/content/sample_severity_data.jsonl'
df_sample.to_json(sample_path, orient='records', lines=True)

print(f"Successfully created a sample dataset with {len(df_sample)} rows.")
print(f"Saved to: {sample_path}")

display(df_sample.head())

Successfully created a sample dataset with 600 rows.
Saved to: /content/sample_severity_data.jsonl


,time,axis,data
0,1.777503e+09,Z,"[-44.8076171875, -49.8076171875, -10.807617187..."
1,1.777503e+09,Z,"[140.0654296875, 218.0654296875, 207.065429687..."
2,1.777503e+09,Y,"[-101.2529296875, -101.2529296875, -98.2529296..."
3,1.777503e+09,Z,"[82.7939453125, 20.7939453125, 4.7939453125, 5..."
4,1.777503e+09,X,"[-24.8525390625, -46.8525390625, -33.852539062..."


In [ ]:
# Dummy data generation removed as we are now using the actual loaded dataset.
print("Using actual dataset loaded in the previous steps.")
print(f"Current dataset shape: {df_severity.shape}")

Using actual dataset loaded in the previous steps.
Current dataset shape: (73363, 6)


In [ ]:
import pandas as pd

# Save the dataframes to pickle format to preserve the list/array data types
raw_save_path = '/content/df_severity_raw.pkl'
merged_save_path = '/content/df_severity_merged.pkl'

print("Saving raw extracted dataframe...")
df_severity.to_pickle(raw_save_path)

if 'df_severity_merged' in locals():
    print("Saving synchronized merged dataframe...")
    df_severity_merged.to_pickle(merged_save_path)

print("\n--- Saved Successfully ---")
print(f"Raw Data: {raw_save_path}")
print(f"Merged Data: {merged_save_path}")

Saving raw extracted dataframe...

--- Saved Successfully ---
Raw Data: /content/df_severity_raw.pkl
Merged Data: /content/df_severity_merged.pkl


In [ ]:
# How to load them later (skip the JSON parsing step):
import pandas as pd
df_severity = pd.read_pickle('/content/df_severity_raw.pkl')
df_severity_merged = pd.read_pickle('/content/df_severity_merged.pkl')

### Phase 2.2b: Target Formulation, Context Extraction & Merging
Extract the target variables, perform Speed Context Normalization using a Min-Max Scaler, and merge the X, Y, and Z rows by exact timestamps.

**Reasoning**:
I will create a Python code block to transform the categorical severity strings into a continuous float scale (0.2, 0.6, 1.0) and merge the isolated sensor rows by exact timestamps.



In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Speed Context Normalization
# Remove 'rpm' suffix and convert to integer
df_severity['motor_speed_int'] = df_severity['motor_speed'].str.replace('rpm', '').astype(int)
speed_scaler = MinMaxScaler()
df_severity['speed_float'] = speed_scaler.fit_transform(df_severity[['motor_speed_int']])

merged_severity_records = []

# Process independent groups
for (state, sev, speed_flt, speed_int), group_df in df_severity.groupby(['machine_state', 'severity', 'speed_float', 'motor_speed_int']):
    group_df = group_df.sort_values('time')

    buffer = {}
    expected_axis = 'X'
    group_time = None

    for _, row in group_df.iterrows():
        axis = row['axis']
        wave = row['data']
        row_time = row['time']

        if axis == 'X' and expected_axis == 'X':
            buffer['X'] = wave
            group_time = row_time
            expected_axis = 'Y'
        elif axis == 'Y' and expected_axis == 'Y':
            buffer['Y'] = wave
            expected_axis = 'Z'
        elif axis == 'Z' and expected_axis == 'Z':
            buffer['Z'] = wave
            merged_severity_records.append({
                'time': group_time,
                'machine_state': state,
                'severity': sev,
                'speed_float': speed_flt,
                'motor_speed_int': speed_int,
                'X': buffer['X'],
                'Y': buffer['Y'],
                'Z': buffer['Z']
            })
            buffer = {}
            expected_axis = 'X'
        else:
            if axis == 'X':
                buffer = {'X': wave}
                group_time = row_time
                expected_axis = 'Y'
            else:
                buffer = {}
                expected_axis = 'X'

df_severity_merged = pd.DataFrame(merged_severity_records)
print(f"Total synchronized samples: {len(df_severity_merged)}")
display(df_severity_merged.head())

Total synchronized samples: 21596


,time,machine_state,severity,speed_float,motor_speed_int,X,Y,Z
0,1.777503e+09,MechanicalLooseness,high,0.0,2200,"[2.7802734375, 6.7802734375, 23.7802734375, 13...","[6.1435546875, 8.1435546875, 15.1435546875, 7....","[-16.3046875, 29.6953125, 70.6953125, 51.69531..."
1,1.777503e+09,MechanicalLooseness,high,0.0,2200,"[-57.8505859375, -59.8505859375, -82.850585937...","[-126.9140625, -139.9140625, -136.9140625, -14...","[57.2138671875, 63.2138671875, 52.2138671875, ..."
2,1.777503e+09,MechanicalLooseness,high,0.0,2200,"[35.51953125, 32.51953125, 31.51953125, 33.519...","[-11.4423828125, -12.4423828125, -7.4423828125...","[46.4111328125, 55.4111328125, 37.4111328125, ..."
3,1.777503e+09,MechanicalLooseness,high,0.0,2200,"[-40.9501953125, -30.9501953125, -34.950195312...","[-140.48828125, -142.48828125, -134.48828125, ...","[63.29296875, 51.29296875, 33.29296875, 24.292..."
4,1.777503e+09,MechanicalLooseness,high,0.0,2200,"[59.89453125, 66.89453125, 78.89453125, 67.894...","[135.0400390625, 139.0400390625, 136.040039062...","[-20.4296875, -30.4296875, -37.4296875, -40.42..."


### Finding Plots

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import scipy.fftpack

fs = 10000  # Assume 10kHz sampling rate

# Get all unique machine states to compare
unique_states = df_severity_merged['machine_state'].unique()

# Create a grid: Rows = Machine States, Cols = 3 (Orbit, FFT, TWF)
subplot_titles = []
for state in unique_states:
    subplot_titles.extend([f"{state} - Orbit (X vs Y)", f"{state} - FFT Spectrum", f"{state} - Time Waveform"])

fig = make_subplots(
    rows=len(unique_states), cols=3,
    subplot_titles=subplot_titles,
    vertical_spacing=0.08,
    horizontal_spacing=0.05
)

colors = ['blue', 'red', 'green', 'purple', 'orange']

for i, state in enumerate(unique_states):
    # Pick one sample per state (we'll grab the first one available)
    sample = df_severity_merged[df_severity_merged['machine_state'] == state].iloc[0]

    x_data = np.array(sample['X'])
    y_data = np.array(sample['Y'])
    time_vector = np.linspace(0, len(x_data) / fs, len(x_data))

    color = colors[i % len(colors)]

    # --- 1. Orbit Plot (X vs Y) ---
    fig.add_trace(go.Scatter(x=x_data, y=y_data, mode='lines', line=dict(color=color, width=1), showlegend=False),
                  row=i+1, col=1)
    fig.update_xaxes(title_text="X Amplitude", row=i+1, col=1)
    fig.update_yaxes(title_text="Y Amplitude", row=i+1, col=1)

    # --- 2. FFT Spectrum (X-axis) ---
    N = len(x_data)
    yf = scipy.fftpack.fft(x_data)
    xf = np.linspace(0.0, 1.0/(2.0*(1/fs)), N//2)
    amplitude = 2.0/N * np.abs(yf[:N//2])

    fig.add_trace(go.Scatter(x=xf, y=amplitude, mode='lines', line=dict(color=color, width=1), showlegend=False),
                  row=i+1, col=2)
    fig.update_xaxes(title_text="Frequency (Hz)", range=[0, 500], row=i+1, col=2)
    fig.update_yaxes(title_text="Amplitude", row=i+1, col=2)

    # --- 3. Time Waveform (X-axis) ---
    fig.add_trace(go.Scatter(x=time_vector, y=x_data, mode='lines', line=dict(color=color, width=1), showlegend=False),
                  row=i+1, col=3)
    fig.update_xaxes(title_text="Time (s)", row=i+1, col=3)
    fig.update_yaxes(title_text="Amplitude", row=i+1, col=3)

# Adjust layout height dynamically based on number of fault states
fig.update_layout(
    title="Comparison of Machine Fault Signatures (Orbit, FFT, TWF)",
    height=300 * len(unique_states),
    width=1200,
    template='plotly_white'
)
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import scipy.fftpack
import scipy.signal

fs = 10000  # Assume 10kHz sampling rate

# Get all unique machine states to compare
unique_states = df_severity_merged['machine_state'].unique()

# Create a grid: Rows = Machine States, Cols = 3 (Orbit, FFT, TWF)
subplot_titles = []
for state in unique_states:
    subplot_titles.extend([f"{state} - Orbit (X vs Y)", f"{state} - FFT Spectrum (Welch)", f"{state} - Time Waveform"])

fig = make_subplots(
    rows=len(unique_states), cols=3,
    subplot_titles=subplot_titles,
    vertical_spacing=0.08,
    horizontal_spacing=0.05
)

colors = ['blue', 'red', 'green', 'purple', 'orange']

for i, state in enumerate(unique_states):
    # Pick one sample per state (we'll grab the first one available)
    sample = df_severity_merged[df_severity_merged['machine_state'] == state].iloc[0]

    x_data = np.array(sample['X'])
    y_data = np.array(sample['Y'])
    time_vector = np.linspace(0, len(x_data) / fs, len(x_data))

    color = colors[i % len(colors)]

    # --- 1. Orbit Plot (X vs Y) - Low-Pass Filtered ---
    # Apply a Butterworth low-pass filter at 100 Hz
    b, a = scipy.signal.butter(N=4, Wn=100, fs=fs, btype='low')
    x_filtered = scipy.signal.filtfilt(b, a, x_data)
    y_filtered = scipy.signal.filtfilt(b, a, y_data)

    fig.add_trace(go.Scatter(x=x_filtered, y=y_filtered, mode='lines', line=dict(color=color, width=1), showlegend=False),
                  row=i+1, col=1)
    fig.update_xaxes(title_text="X Amplitude", row=i+1, col=1)
    fig.update_yaxes(title_text="Y Amplitude", row=i+1, col=1)

    # --- 2. FFT Spectrum (X-axis) - Welch's Method ---
    # Use Welch's method for a smoother periodogram
    f, Pxx = scipy.signal.welch(x_data, fs, nperseg=min(1024, len(x_data)))
    amplitude_welch = np.sqrt(Pxx) # Convert power to amplitude

    fig.add_trace(go.Scatter(x=f, y=amplitude_welch, mode='lines', line=dict(color=color, width=1), showlegend=False),
                  row=i+1, col=2)
    fig.update_xaxes(title_text="Frequency (Hz)", range=[0, 500], row=i+1, col=2)
    fig.update_yaxes(title_text="Amplitude", row=i+1, col=2)

    # --- 3. Time Waveform (X-axis) ---
    fig.add_trace(go.Scatter(x=time_vector, y=x_data, mode='lines', line=dict(color=color, width=1), showlegend=False),
                  row=i+1, col=3)
    fig.update_xaxes(title_text="Time (s)", row=i+1, col=3)
    fig.update_yaxes(title_text="Amplitude", row=i+1, col=3)

# Adjust layout height dynamically based on number of fault states
fig.update_layout(
    title="Comparison of Machine Fault Signatures (Filtered Orbit, Welch FFT, TWF)",
    height=300 * len(unique_states),
    width=1200,
    template='plotly_white'
)
fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np
import scipy.fftpack
import pandas as pd

fs = 10000  # 10kHz sampling rate

# ==========================================
# 1. 3D Waterfall (Cascade) Plot (Unbalance)
# ==========================================
target_state = 'Unbalance'
waterfall_df = df_severity_merged[df_severity_merged['machine_state'] == target_state]

fig_waterfall = go.Figure()
speeds = sorted(waterfall_df['motor_speed_int'].unique())

for speed in speeds:
    # Get one sample per speed
    sample = waterfall_df[waterfall_df['motor_speed_int'] == speed].iloc[0]
    x_data = np.array(sample['X'])

    N = len(x_data)
    yf = scipy.fftpack.fft(x_data)
    xf = np.linspace(0.0, 1.0/(2.0*(1/fs)), N//2)
    amplitude = 2.0/N * np.abs(yf[:N//2])

    # Filter to look at relevant low frequencies (<500 Hz)
    freq_mask = xf < 500

    fig_waterfall.add_trace(go.Scatter3d(
        x=xf[freq_mask],
        y=[speed] * sum(freq_mask),
        z=amplitude[freq_mask],
        mode='lines',
        name=f'{speed} RPM',
        line=dict(width=4)
    ))

fig_waterfall.update_layout(
    title=f"Plot D: 3D Waterfall / Cascade Plot - {target_state}",
    scene=dict(
        xaxis_title='Frequency (Hz)',
        yaxis_title='Motor Speed (RPM)',
        zaxis_title='Amplitude'
    ),
    width=900, height=600
)
fig_waterfall.show()

# ==========================================
# 2. Bode-Style Plot (RPM vs RMS Amplitude)
# ==========================================
def calculate_rms(data_list):
    return np.sqrt(np.mean(np.array(data_list)**2))

# Sample a subset to keep the scatter plot rendering fast (e.g., 1000 points)
plot_df = df_severity_merged.sample(min(1000, len(df_severity_merged)), random_state=42).copy()
plot_df['X_RMS'] = plot_df['X'].apply(calculate_rms)

fig_bode = go.Figure()
for state in plot_df['machine_state'].unique():
    state_df = plot_df[plot_df['machine_state'] == state]
    fig_bode.add_trace(go.Scatter(
        x=state_df['motor_speed_int'] + np.random.normal(0, 15, len(state_df)), # Add slight jitter for visibility
        y=state_df['X_RMS'],
        mode='markers',
        name=state,
        marker=dict(size=8, opacity=0.7)
    ))

# --- Handle Outliers ---
# Calculate the 95th percentile of RMS to set a sensible Y-axis max limit
y_max = np.percentile(plot_df['X_RMS'], 95) * 1.5

fig_bode.update_layout(
    title="Plot F: Bode-Style Scatter (RPM vs. Overall Vibration RMS)",
    xaxis_title="Motor Speed (RPM)",
    yaxis_title="Overall Vibration RMS (X-Axis)",
    yaxis=dict(range=[0, y_max]), # Zoom in to ignore extreme outliers
    width=900, height=500,
    template='plotly_white'
)
fig_bode.show()